<a href="https://colab.research.google.com/github/aravindchandra/Aravind_INFO5731_Spring2026/blob/main/In_Class_Exercise_5%266_Feature_Extraction_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# In-Class Assignment — Python for Feature Extraction
**Time:** 20 minutes  |  **Points:** 20


....


Dataset file: `product_reviews.txt`

## Load the dataset
Download it from **Canvas**, then run the upload cell below to select the file from your computer.


In [1]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()   # choose product_reviews.txt
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving product_reviews.txt to product_reviews (1).txt
Uploaded file name: product_reviews (1).txt


## Q1 (2 point) — Load & Read & inspect

## Note about the header
**Important:** The dataset file includes a **header row** (column names).  
Make sure the header is used as the column names — it **should NOT appear as a data row** in your DataFrame.

Tip: Use the filename variable printed above when reading the file.
After Reading, check that your columns are `id` and `text`.
Print: **(a)** `df.shape`, **(b)** `df.head(3\)`.  


In [2]:
import pandas as pd

# Load the dataset (TAB-separated, with header)
filename = "product_reviews.txt"
df = pd.read_csv(filename, sep='\t', header=0)

# Print shape
print(df.shape)

# Print first 3 rows
print(df.head(3))



(10, 2)
   id                                               text
0   1  Love this blender! Smoothies are super creamy ...
1   2  Terrible quality... stopped working after 2 da...
2   3      Good value for the price. Shipping was quick.


## Q2 (4 points) — Basic handcrafted features  
Create these columns and then display the DataFrame:
- `word_count` = number of words  
- `char_count` = number of characters  
- `avg_word_len` = average word length (ignore punctuation)  
- `excl_count` = number of `!` characters  

Print: `id, word_count, char_count, avg_word_len, excl_count`.


In [4]:
import re

# word_count = number of words
df['word_count'] = df['text'].apply(lambda x: len(x.split()))

# char_count = number of characters
df['char_count'] = df['text'].apply(len)

# avg_word_len = average word length (ignore punctuation)
df['avg_word_len'] = df['text'].apply(
    lambda x: sum(len(re.sub(r'[^\w]', '', word)) for word in x.split()) / len(x.split())
)

# excl_count = number of '!' characters
df['excl_count'] = df['text'].apply(lambda x: x.count('!'))

# Print required columns
print(df[['id', 'word_count', 'char_count', 'avg_word_len', 'excl_count']])


   id  word_count  char_count  avg_word_len  excl_count
0   1           9          55      5.000000           1
1   2           7          51      5.571429           3
2   3           8          45      4.500000           0
3   4          10          56      4.500000           0
4   5           8          53      5.500000           1
5   6          10          51      4.000000           0
6   7           9          50      4.444444           0
7   8           6          40      5.500000           0
8   9           7          52      6.285714           0
9  10           7          48      5.428571           0


## Q3 (6 points) — Bag-of-Words (CountVectorizer)  
Use `CountVectorizer(stop_words="english")` on `df["text"]`. Print:
1) vocabulary size (number of features)  
2) top 10 words by **total count** across all documents (word + count)


In [5]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Initialize vectorizer with English stopwords
vectorizer = CountVectorizer(stop_words='english')

# Fit and transform the text data
X = vectorizer.fit_transform(df['text'])

# Vocabulary size
vocab_size = len(vectorizer.get_feature_names_out())
print("Vocabulary size:", vocab_size)

# Sum of word counts across all documents
word_counts = np.array(X.sum(axis=0)).flatten()

# Get words
words = vectorizer.get_feature_names_out()

# Combine words and counts, then sort
word_freq = list(zip(words, word_counts))
word_freq_sorted = sorted(word_freq, key=lambda x: x[1], reverse=True)

# Print top 10 words
print("\nTop 10 words by total count:")
for word, count in word_freq_sorted[:10]:
    print(word, count)


Vocabulary size: 50

Top 10 words by total count:
amazing 1
app 1
battery 1
blender 1
box 1
buy 1
charged 1
clear 1
crashing 1
creamy 1


## Q4 (4 points) — Bigram features  
Use `CountVectorizer(stop_words="english", ngram_range=(2,2))`.  
Print the top 5 bigrams by total count (bigram + count).


In [6]:
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

# Initialize CountVectorizer for bigrams only
vectorizer_bigram = CountVectorizer(stop_words='english', ngram_range=(2, 2))

# Fit and transform
X_bigram = vectorizer_bigram.fit_transform(df['text'])

# Get bigram names
bigrams = vectorizer_bigram.get_feature_names_out()

# Sum counts across all documents
bigram_counts = np.array(X_bigram.sum(axis=0)).flatten()

# Combine and sort
bigram_freq = list(zip(bigrams, bigram_counts))
bigram_freq_sorted = sorted(bigram_freq, key=lambda x: x[1], reverse=True)

# Print top 5 bigrams
print("Top 5 bigrams:")
for bigram, count in bigram_freq_sorted[:5]:
    print(bigram, count)

Top 5 bigrams:
amazing screen 1
app keeps 1
battery life 1
blender smoothies 1
box damaged 1


## Q5 (4 points) — TF-IDF features  
Use `TfidfVectorizer(stop_words="english", ngram_range=(1,2))`.  
Compute the **average TF-IDF** score of each term across documents and print the top 5 terms (term + avg score).


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# Initialize TF-IDF Vectorizer (unigrams + bigrams)
tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2))

# Fit and transform
X_tfidf = tfidf.fit_transform(df['text'])

# Get feature names
terms = tfidf.get_feature_names_out()

# Compute average TF-IDF score for each term
avg_tfidf = np.array(X_tfidf.mean(axis=0)).flatten()

# Combine terms and scores
term_scores = list(zip(terms, avg_tfidf))

# Sort by score (descending)
term_scores_sorted = sorted(term_scores, key=lambda x: x[1], reverse=True)

# Print top 5 terms
print("Top 5 terms by average TF-IDF score:")
for term, score in term_scores_sorted[:5]:
    print(term, round(score, 4))



Top 5 terms by average TF-IDF score:
clear 0.0378
easy 0.0378
easy instructions 0.0378
instructions 0.0378
instructions clear 0.0378


## Grading Checklist
- Q1: correct prints  
- Q2: correct feature columns + requested display  
- Q3: correct vocabulary size + correct top 10 words by total count  
- Q4: correct top 5 bigrams by total count  
- Q5: correct top 5 TF-IDF terms by average score
